# BNPL Credit Scorecard — EDA

Exploratory analysis of the cleaned training data `data/processed/train.csv`.
The project root is located automatically, so this notebook runs regardless of its working directory.

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Locate the project root by walking up to the directory containing src/config.py
def find_root():
    cwd = Path.cwd()
    for d in [cwd, *cwd.parents]:
        if (d / "src" / "config.py").exists():
            return d
    raise FileNotFoundError("Project root not found; make sure the notebook is inside the project")

ROOT = find_root()
print(f"project root: {ROOT}")

df = pd.read_csv(ROOT / "data" / "processed" / "train.csv")
print(f"shape: {df.shape}")
print(f"columns: {list(df.columns)}")
df.head()

In [ ]:
print(f"bad rate = {df['bad'].mean():.2%}")
missing = df.isna().sum()
print("missing values (>0):")
print(missing[missing > 0])

In [ ]:
num_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico', 'credit_age_years']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, c in zip(axes.ravel(), num_cols):
    df[c].hist(ax=ax, bins=40, color='#4a86c8')
    ax.set_title(c)
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['term', 'home_ownership', 'verification_status', 'purpose']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, c in zip(axes.ravel(), cat_cols):
    df.groupby(c)['bad'].mean().sort_values().plot(kind='barh', ax=ax, color='#4a86c8')
    ax.set_title(f'bad rate by {c}')
    ax.set_xlabel('bad rate')
plt.tight_layout()
plt.show()

## Takeaways

- Training bad rate is ~20% (includes the 2008-2011 high-default crisis vintages);
- int_rate / FICO / DTI show a clear monotonic relationship with bad rate (matching the WoE monotonicity);
- Missing values concentrate in dti / annual_inc / credit_age_years, handled as a separate MISSING bin in WoE.